# 01 — PostgreSQL Telemetry SQL Walkthrough

Purpose:
This is the simple starting notebook for the PostgreSQL telemetry lab.

It introduces:
- connecting to PostgreSQL
- running simple SELECT queries
- inspecting tables
- previewing telemetry data
- filtering rows
- ordering rows
- reading simple JSONB tags
- doing one basic JOIN to make service IDs readable

This notebook is intentionally simple.
Deeper topics are in notebooks 02 through 06.


## Cell 2 — Install/import dependencies


In [1]:
import pandas as pd
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

print("Imports loaded.")


Imports loaded.


## Cell 3 — Connection settings


In [2]:
DB_HOST = "host.docker.internal"
DB_PORT = 5432
DB_NAME = "studybook"
DB_USER = "sb_user"
DB_PASSWORD = "sb_pass_123"

password_encoded = quote_plus(DB_PASSWORD)

DATABASE_URL = (
    f"postgresql+psycopg2://{DB_USER}:{password_encoded}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

engine = create_engine(DATABASE_URL)

print("Database URL created.")

# If this notebook runs directly on Windows instead of inside a container,
# change DB_HOST to "localhost".


ModuleNotFoundError: No module named 'psycopg2'

## Cell 4 — Smoke test connection


In [ ]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT current_database(), current_user, now();"))
    row = result.fetchone()

row


## Cell 5 — Helper function to run SQL


In [ ]:
def run_sql(sql: str) -> pd.DataFrame:
    """
    Run SQL against the local PostgreSQL telemetry lab
    and return the result as a pandas DataFrame.
    """
    with engine.connect() as conn:
        return pd.read_sql_query(text(sql), conn)


## Cell 6 — Helper function to inspect one table safely


In [ ]:
def inspect_table_safe(table_name: str) -> None:
    """
    Safely inspect a table without changing data.
    Shows column metadata, row count, and a small preview.
    """
    metadata_sql = f"""
    SELECT
        table_schema,
        table_name,
        ordinal_position,
        column_name,
        data_type,
        is_nullable,
        column_default
    FROM information_schema.columns
    WHERE table_schema = 'public'
      AND table_name = '{table_name}'
    ORDER BY ordinal_position;
    """

    count_sql = f"""
    SELECT COUNT(*) AS row_count
    FROM {table_name};
    """

    preview_sql = f"""
    SELECT *
    FROM {table_name}
    LIMIT 10;
    """

    print(f"Column metadata for public.{table_name}")
    display(run_sql(metadata_sql))

    print(f"Row count for public.{table_name}")
    display(run_sql(count_sql))

    print(f"Preview rows from public.{table_name}")
    display(run_sql(preview_sql))


# 01 — Basic SELECT Practice


## 01.1 Verify core tables exist

Before querying data, verify which tables are available in the public schema.


In [ ]:
sql = """
SELECT
    table_name
FROM information_schema.tables
WHERE table_schema = 'public'
ORDER BY table_name;
"""

run_sql(sql)


## 01.2 Inspect telemetry_samples

This table stores sampled telemetry metrics such as CPU, memory, latency,
request rate, error rate, cost, and tags.


In [ ]:
inspect_table_safe("telemetry_samples")


## 01.3 Preview telemetry rows

Start with a small preview of the main telemetry table.


In [ ]:
sql = """
SELECT
    sample_id,
    sampled_at,
    service_id,
    host_id,
    cpu_utilization_pct,
    memory_utilization_pct,
    p95_latency_ms,
    requests_per_min,
    error_rate_pct,
    cloud_cost_usd,
    tags
FROM telemetry_samples
ORDER BY sampled_at
LIMIT 20;
"""

run_sql(sql)


## 01.4 Select only CPU and memory columns

A SELECT statement can return only the columns needed for the question.


In [ ]:
sql = """
SELECT
    sampled_at,
    service_id,
    host_id,
    cpu_utilization_pct,
    memory_utilization_pct
FROM telemetry_samples
ORDER BY sampled_at
LIMIT 20;
"""

run_sql(sql)


## 01.5 Filter high CPU samples

WHERE filters rows.
This query finds samples where CPU utilization is at least 80%.


In [ ]:
sql = """
SELECT
    sampled_at,
    service_id,
    host_id,
    cpu_utilization_pct,
    memory_utilization_pct
FROM telemetry_samples
WHERE cpu_utilization_pct >= 80
ORDER BY cpu_utilization_pct DESC
LIMIT 20;
"""

run_sql(sql)


## 01.6 Filter high memory samples

This query finds samples where memory utilization is at least 80%.


In [ ]:
sql = """
SELECT
    sampled_at,
    service_id,
    host_id,
    cpu_utilization_pct,
    memory_utilization_pct
FROM telemetry_samples
WHERE memory_utilization_pct >= 80
ORDER BY memory_utilization_pct DESC
LIMIT 20;
"""

run_sql(sql)


## 01.7 Filter high latency samples

This query finds samples with high P95 latency.
The column p95_latency_ms is already a sampled P95 metric.


In [ ]:
sql = """
SELECT
    sampled_at,
    service_id,
    host_id,
    p95_latency_ms,
    requests_per_min,
    error_rate_pct
FROM telemetry_samples
WHERE p95_latency_ms >= 400
ORDER BY p95_latency_ms DESC
LIMIT 20;
"""

run_sql(sql)


## 01.8 Use AND / OR conditions

AND means both conditions must be true.
OR means either condition can be true.


In [ ]:
sql = """
SELECT
    sampled_at,
    service_id,
    host_id,
    cpu_utilization_pct,
    memory_utilization_pct,
    p95_latency_ms,
    error_rate_pct
FROM telemetry_samples
WHERE cpu_utilization_pct >= 75
   OR memory_utilization_pct >= 75
   OR p95_latency_ms >= 400
ORDER BY sampled_at
LIMIT 30;
"""

run_sql(sql)


## 01.9 Preview services lookup table

The telemetry table stores service_id.
The services table makes that ID readable as service_name.


In [ ]:
sql = """
SELECT
    service_id,
    service_name
FROM services
ORDER BY service_id;
"""

run_sql(sql)


## 01.10 First simple JOIN

This JOIN connects telemetry_samples to services.
Now service_id becomes readable as service_name.

JOIN by itself means INNER JOIN.


In [ ]:
sql = """
SELECT
    t.sampled_at,
    s.service_name,
    t.host_id,
    t.cpu_utilization_pct,
    t.memory_utilization_pct,
    t.p95_latency_ms
FROM telemetry_samples t
JOIN services s
    ON s.service_id = t.service_id
ORDER BY
    t.sampled_at,
    s.service_name
LIMIT 20;
"""

run_sql(sql)


## 01.11 Preview JSONB tags

The tags column is JSONB.
It stores flexible metadata inside one column.
Use ->> to extract a JSON value as text.


In [ ]:
sql = """
SELECT
    sample_id,
    sampled_at,
    tags
FROM telemetry_samples
ORDER BY sampled_at
LIMIT 20;
"""

run_sql(sql)


## 01.12 Extract simple JSONB tag values

tags ->> 'key_name' extracts a value from the JSONB tags column as text.
If a key does not exist, PostgreSQL returns NULL.

This lab data uses the keys `team`, `env`, and `region`.


In [ ]:
sql = """
SELECT
    sample_id,
    sampled_at,
    tags,
    tags ->> 'team' AS tag_team,
    tags ->> 'env' AS tag_env,
    tags ->> 'region' AS tag_region
FROM telemetry_samples
ORDER BY sampled_at
LIMIT 20;
"""

run_sql(sql)


## 01.13 First simple GROUP BY

GROUP BY collapses many telemetry rows into one summary row per service.
This is only a beginner preview. Deeper aggregation is in notebook 03.


In [ ]:
sql = """
SELECT
    s.service_name,
    ROUND(AVG(t.cpu_utilization_pct), 2) AS avg_cpu_pct,
    ROUND(AVG(t.memory_utilization_pct), 2) AS avg_memory_pct
FROM telemetry_samples t
JOIN services s
    ON s.service_id = t.service_id
GROUP BY s.service_name
ORDER BY avg_cpu_pct DESC;
"""

run_sql(sql)


## 01.14 What to study next

Notebook 01 is only the ramp-up.

Next notebooks:
- 02_joins_and_group_by.ipynb — joins and grouped summaries
- 03_capacity_aggregation.ipynb — capacity rollups and P95
- 04_window_functions.ipynb — ROW_NUMBER, RANK, LAG, LEAD, moving averages
- 05_interview_questions.ipynb — interview-style SQL questions
- 06_server_rollup_paractice.ipynb — capstone-style 5-minute server telemetry rollup


In [2]:
! pip install sqlalchemy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 3.4 MB/s  0:00:00m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.4/611.4 kB 1.8 MB/s  0:00:00m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [sqlalchemy]2 [sqlalchemy]


In [3]:
!pip install pymysql
